# 04 — Residencial en regiones aisladas (Insular / Sureste)

**Caso especial, NO integrado al flujo general de regionalización.** Se corre de forma
independiente, después del flujo general, con los mismos insumos que el resto del flujo:
el **SAND nacional** y `Insumos/Mapeo/participaciones.xlsx` (único archivo de
participaciones; los `Insumos/Participacion_*.xlsx` son la materia prima de
`00_Generar_Mapeo.ipynb`, no entrada de este notebook).

En el escenario regional las 5 regiones conectadas conservan el patrón nacional
`..._URB` / `..._RUR` (tecnologías `DEMRES*`, fuels `RES*`). En **IN** y **SE**, por ser
aisladas, el modelo se simplificó: se sumó URB+RUR, se quitó el sufijo y algunas
tecnologías no existen. Este notebook recompone la demanda/actividad residencial de IN/SE:

    valor_IN(base) = Σ_sufijo  nacional(base_sufijo) × participación_IN(base_sufijo)

colapsando URB+RUR sobre el **valor absoluto** del nacional. Reglas de destino (en orden):

| Situación | Destino |
|---|---|
| El código sin sufijo existe en la región | se acumula ahí (colapso URB+RUR) |
| No existe, tecnología eléctrica (`DEMRESELC`) | `DEMRESELCOTH` conservando eficiencia (`_HIG/_LOW`; otras → `_MID`) |
| No existe, fuel `RESILU/RESTV/RESWHT/RESWSH` | `RESOTH` |

`PARAMETROS_A_CORREGIR` (celda de configuración) fija qué parámetros se recomponen, igual
que `PARAMETROS_A_REGIONALIZAR` en el notebook 02; la dimensión de cada uno (TECHNOLOGY o
FUEL) se deduce del config de otoole.

Salidas (el nacional y las participaciones originales **nunca se modifican**):
`SANDs_Reducidos/SAND_Residencial_IN_SE.xlsx`, `reportes/residencial_in_se_log.csv`
(auditoría) y `reportes/participaciones_residencial_corregido.csv` (participación efectiva).

## Configuración de rutas

In [1]:
# --- Setup y rutas (editar aquí) ---
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.6f}")

RAIZ = Path.cwd().resolve()
if not (RAIZ / "config").exists():   # ejecutado desde notebooks/
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import residencial_in_se as res_inse
import sand_io
import utils
import yaml_parser
utils.configurar_logging()

SAND_NACIONAL        = RAIZ / "SAND_Nacional_base" / "scenario_CN_Nacional_166_Parameters_SAND.xlsx"
CONFIG_OTOOLE        = RAIZ / "template_config.yaml"
# Único insumo de participaciones del flujo (salida de 00_Generar_Mapeo.ipynb).
RUTA_PARTICIPACIONES = RAIZ / "Insumos" / "Mapeo" / "participaciones.xlsx"
DIR_SANDS_REDUCIDOS  = RAIZ / "SANDs_Reducidos"
DIR_REPORTES         = RAIZ / "reportes"

# Parámetros a corregir en IN/SE (equivalente a PARAMETROS_A_REGIONALIZAR del 02).
# La demanda residencial de fuels vive en SpecifiedAnnualDemand (AccumulatedAnnualDemand
# está en 0); la actividad de tecnologías en TotalTechnologyAnnualActivityLowerLimit.
PARAMETROS_A_CORREGIR = ["ResidualCapacity","AccumulatedAnnualDemand","TotalAnnualMinCapacityInvestment","TotalAnnualMaxCapacityInvestment","TotalTechnologyAnnualActivityLowerLimit","TotalTechnologyAnnualActivityUpperLimit","SpecifiedAnnualDemand","SpecifiedDemandProfile"]
ANIO_MAX = 2054            # 2055 se excluye (convención del proyecto)
MODO_DRY_RUN = False       # True => no escribe archivos, solo muestra qué haría

otoole = yaml_parser.cargar_config_otoole(CONFIG_OTOOLE)
# Dimensión de código de cada parámetro (TECHNOLOGY si se indexa por tecnología, si no
# FUEL): la misma convención con la que participaciones.xlsx separa sus dos columnas.
DIMENSIONES = {p: res_inse.dimension_parametro(p, otoole["param"])
               for p in PARAMETROS_A_CORREGIR}

print("Regiones aisladas:", res_inse.REGIONES_AISLADAS)
print("Participaciones  :", RUTA_PARTICIPACIONES.relative_to(RAIZ))
print("Parámetros       :", DIMENSIONES)

Regiones aisladas: {'Insular': 'IN', 'Este': 'SE'}
Participaciones  : Insumos\Mapeo\participaciones.xlsx
Parámetros       : {'ResidualCapacity': 'TECHNOLOGY', 'AccumulatedAnnualDemand': 'FUEL', 'TotalAnnualMinCapacityInvestment': 'TECHNOLOGY', 'TotalAnnualMaxCapacityInvestment': 'TECHNOLOGY', 'TotalTechnologyAnnualActivityLowerLimit': 'TECHNOLOGY', 'TotalTechnologyAnnualActivityUpperLimit': 'TECHNOLOGY', 'SpecifiedAnnualDemand': 'FUEL', 'SpecifiedDemandProfile': 'FUEL'}


## 1. Cargar insumos

In [2]:
df_nacional = sand_io.cargar_sand(SAND_NACIONAL)
ANIOS = [int(c) for c in sand_io.columnas_anio(df_nacional) if int(c) <= ANIO_MAX]
# Existencia real de códigos en el escenario regional (misma base de mapeo_maestro.csv)
existentes = (set(pd.read_csv(RAIZ / "CSV_Regional" / "TECHNOLOGY.csv")["VALUE"].astype(str))
              | set(pd.read_csv(RAIZ / "CSV_Regional" / "FUEL.csv")["VALUE"].astype(str)))

# Participaciones de IN/SE por parámetro, desde el único archivo de participaciones.
# `cargar_participacion` descarta las filas cuya dimensión no aplica al parámetro.
participaciones = {
    p: res_inse.cargar_participacion(RUTA_PARTICIPACIONES, p, dim,
                                     params_otoole=otoole["param"], anios=ANIOS)
    for p, dim in DIMENSIONES.items()
}

print(f"SAND nacional: {len(df_nacional):,} filas | años {ANIOS[0]}–{ANIOS[-1]} | "
      f"códigos regionales existentes: {len(existentes):,}")
for p, pct in participaciones.items():
    print(f"  {p} ({DIMENSIONES[p]}): {len(pct):,} filas IN/SE, "
          f"{pct['codigo'].nunique()} códigos")

12:38:02 INFO    regionalizador: participaciones.xlsx: formato ancho en regiones detectado (7 columnas de región)
12:38:02 INFO    regionalizador: participaciones.xlsx: columna 'Parámetro' vacía; se infiere de la lista de parámetros
12:38:02 INFO    regionalizador: participaciones.xlsx: sin columna 'Parámetro'; se aplica a ['ResidualCapacity']
12:38:02 WARNING regionalizador: participaciones.xlsx: ResidualCapacity no se indexa por FUEL; se descartan 3696 filas de participación
12:38:02 INFO    regionalizador: Participaciones cargadas: 24486 filas (106 combos)
12:38:02 INFO    residencial_in_se: ResidualCapacity: 6996 participaciones IN/SE (106 códigos)
12:38:03 INFO    regionalizador: participaciones.xlsx: formato ancho en regiones detectado (7 columnas de región)
12:38:03 INFO    regionalizador: participaciones.xlsx: columna 'Parámetro' vacía; se infiere de la lista de parámetros
12:38:03 INFO    regionalizador: participaciones.xlsx: sin columna 'Parámetro'; se aplica a ['AccumulatedA

## 2. Corregir residencial de IN / SE

In [3]:
sands, logs, pcts = [], [], []
for parametro, dim in DIMENSIONES.items():
    df_sand, df_log, df_pct = res_inse.corregir_parametro(
        df_nacional, participaciones[parametro], parametro, dim, existentes,
        anio_max=ANIO_MAX)
    print(f"\n{parametro} ({dim}): {len(df_sand)} filas IN/SE, {len(df_log)} reasignaciones")
    display(df_log["Regla"].value_counts())
    sands.append(df_sand); logs.append(df_log); pcts.append(df_pct)

sand_corregido = pd.concat(sands, ignore_index=True)
log_corregido = pd.concat(logs, ignore_index=True)
pct_corregido = pd.concat(pcts, ignore_index=True)
display(sand_corregido.head(20))

12:38:14 INFO    residencial_in_se: ResidualCapacity: 50 filas SAND para IN/SE, 140 reasignaciones (0 sin regla)

ResidualCapacity (TECHNOLOGY): 50 filas IN/SE, 140 reasignaciones


Regla
colapso_sin_sufijo                  79
redirigir_electrica_DEMRESELCOTH    61
Name: count, dtype: int64

12:38:15 INFO    residencial_in_se: AccumulatedAnnualDemand: 8 filas SAND para IN/SE, 26 reasignaciones (0 sin regla)

AccumulatedAnnualDemand (FUEL): 8 filas IN/SE, 26 reasignaciones


Regla
colapso_sin_sufijo     13
agregar_fuel_RESOTH    13
Name: count, dtype: int64

12:38:17 INFO    residencial_in_se: TotalAnnualMinCapacityInvestment: 50 filas SAND para IN/SE, 140 reasignaciones (0 sin regla)

TotalAnnualMinCapacityInvestment (TECHNOLOGY): 50 filas IN/SE, 140 reasignaciones


...

12:38:25 INFO    residencial_in_se: SpecifiedAnnualDemand: 8 filas SAND para IN/SE, 26 reasignaciones (0 sin regla)

SpecifiedAnnualDemand (FUEL): 8 filas IN/SE, 26 reasignaciones


Regla
colapso_sin_sufijo     13
agregar_fuel_RESOTH    13
Name: count, dtype: int64

12:38:26 INFO    residencial_in_se: SpecifiedDemandProfile: 8 filas SAND para IN/SE, 26 reasignaciones (0 sin regla)

SpecifiedDemandProfile (FUEL): 8 filas IN/SE, 26 reasignaciones


Regla
colapso_sin_sufijo     13
agregar_fuel_RESOTH    13
Name: count, dtype: int64

,Parameter,REGION,TECHNOLOGY,EMISSION,MODE_OF_OPERATION,FUEL,TIMESLICE,STORAGE,REGION2,Time indipendent variables,...,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054
0,ResidualCapacity,RE1,IN_DEMRESELCAIR_PAR_LOW,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,0.000602,0.000343,0.000177,0.000082,0.000033,0.000012,0.000004,0.000001,0.000000,0.000000
1,ResidualCapacity,RE1,IN_DEMRESELCAIR_PAR_MID,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,0.000014,0.000008,0.000004,0.000002,0.000001,0.000000,0.000000,0.000000,0.000000,0.000000
2,ResidualCapacity,RE1,IN_DEMRESELCAIR_POR_HIG,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,ResidualCapacity,RE1,IN_DEMRESELCAIR_POR_LOW,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,ResidualCapacity,RE1,IN_DEMRESELCAIR_POR_MID,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,ResidualCapacity,RE1,IN_DEMRESELCAIR_SPL_HIG,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,ResidualCapacity,RE1,IN_DEMRESELCAIR_SPL_LOW,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,ResidualCapacity,RE1,IN_DEMRESELCAIR_SPL_MID,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,ResidualCapacity,RE1,IN_DEMRESELCCKN_HIG,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,0.003518,0.003741,0.003967,0.004194,0.004418,0.004626,0.004765,0.004903,0.005031,0.005158
9,ResidualCapacity,RE1,IN_DEMRESELCCKN_LOW,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,0.002621,0.002337,0.002077,0.001838,0.001619,0.001421,0.001243,0.001082,0.000939,0.000812


## 3. Validación y resumen de auditoría

In [4]:
# ¿Quedó algún código con participación en IN/SE sin regla de destino?
sin_regla = log_corregido[log_corregido["Regla"] == res_inse.REGLA_SIN_REGLA]
if sin_regla.empty:
    print("OK: todos los códigos con participación en IN/SE fueron reubicados (0 sin regla).")
else:
    display(Markdown(f"**{len(sin_regla)} casos SIN REGLA — revisar manualmente:**"))
    display(sin_regla)

display(Markdown("**Reasignaciones por región y regla:**"))
display(log_corregido.groupby(["Region", "Regla"]).size().unstack(fill_value=0))

# Chequeo de conservación: Σ del SAND corregido == Σ (nacional × participación IN/SE)
cols_anio = [str(a) for a in ANIOS]
for parametro, dim in DIMENSIONES.items():
    pct = participaciones[parametro]
    nac_p = df_nacional[df_nacional["Parameter"] == parametro]
    val_nac = (nac_p.assign(_c=nac_p[dim].astype(str))
               .groupby("_c")[cols_anio].apply(lambda g: g.apply(pd.to_numeric, errors="coerce").sum()))
    esperado = 0.0
    for _, r in pct[pct["Participacion"] > 0].iterrows():
        c, a = str(r["codigo"]), int(r["Anio"])
        if c in val_nac.index and str(a) in cols_anio:
            v = val_nac.loc[c, str(a)]
            esperado += (0 if pd.isna(v) else v) * r["Participacion"]
    gen = (sand_corregido[sand_corregido["Parameter"] == parametro][cols_anio]
           .apply(pd.to_numeric, errors="coerce").sum().sum())
    print(f"Conservación {parametro}: generado={gen:,.4f}  esperado(Σ nac×%)={esperado:,.4f}  "
          f"dif={abs(gen - esperado):.2e}")

OK: todos los códigos con participación en IN/SE fueron reubicados (0 sin regla).


**Reasignaciones por región y regla:**

Regla,agregar_fuel_RESOTH,colapso_sin_sufijo,redirigir_electrica_DEMRESELCOTH
Region,,,
IN,15,145,120
SE,24,289,185


Conservación ResidualCapacity: generado=13.2721  esperado(Σ nac×%)=13.2721  dif=6.37e-07
Conservación AccumulatedAnnualDemand: generado=0.0000  esperado(Σ nac×%)=0.0000  dif=0.00e+00
Conservación TotalAnnualMinCapacityInvestment: generado=0.1186  esperado(Σ nac×%)=0.1186  dif=5.28e-08
Conservación TotalAnnualMaxCapacityInvestment: generado=1,016,909.0994  esperado(Σ nac×%)=1,016,909.0994  dif=2.34e-05
Conservación TotalTechnologyAnnualActivityLowerLimit: generado=12.5128  esperado(Σ nac×%)=12.5128  dif=7.00e-07
Conservación TotalTechnologyAnnualActivityUpperLimit: generado=15,219,082.6366  esperado(Σ nac×%)=15,219,082.6366  dif=1.52e-05
Conservación SpecifiedAnnualDemand: generado=40.8830  esperado(Σ nac×%)=40.8830  dif=2.36e-06
Conservación SpecifiedDemandProfile: generado=7.3030  esperado(Σ nac×%)=7.3030  dif=3.23e-06


## 4. Exportar SAND corregido, log de auditoría y participación efectiva

In [5]:
RUTA_SAND = DIR_SANDS_REDUCIDOS / "SAND_Residencial_IN_SE.xlsx"
RUTA_LOG  = DIR_REPORTES / "residencial_in_se_log.csv"
RUTA_PCT  = DIR_REPORTES / "participaciones_residencial_corregido.csv"

if MODO_DRY_RUN:
    display(Markdown("**DRY-RUN — no se escribió nada.** Se habría generado:"))
    print(f"  {RUTA_SAND}  ({len(sand_corregido)} filas)")
    print(f"  {RUTA_LOG}  ({len(log_corregido)} filas)")
    print(f"  {RUTA_PCT}  ({len(pct_corregido)} filas)")
else:
    DIR_SANDS_REDUCIDOS.mkdir(parents=True, exist_ok=True)
    DIR_REPORTES.mkdir(parents=True, exist_ok=True)
    sand_io.escribir_sand(sand_corregido, RUTA_SAND)
    log_corregido.to_csv(RUTA_LOG, index=False, encoding="utf-8-sig")
    pct_corregido.to_csv(RUTA_PCT, index=False, encoding="utf-8-sig")
    print(f"SAND corregido : {RUTA_SAND.relative_to(RAIZ)}  ({len(sand_corregido)} filas)")
    print(f"Log auditoría  : {RUTA_LOG.relative_to(RAIZ)}  ({len(log_corregido)} filas)")
    print(f"Part. efectiva : {RUTA_PCT.relative_to(RAIZ)}  ({len(pct_corregido)} filas)")

print("\nEl SAND nacional y las participaciones originales NO se modificaron.")

SAND corregido : SANDs_Reducidos\SAND_Residencial_IN_SE.xlsx  (274 filas)
Log auditoría  : reportes\residencial_in_se_log.csv  (778 filas)
Part. efectiva : reportes\participaciones_residencial_corregido.csv  (4998 filas)

El SAND nacional y las participaciones originales NO se modificaron.
